# Phase 1: Data Processing for Twi Voice Technology (Google Colab)

This notebook processes the datasets in Google Drive:
- Extracts archives (JW300, Common Voice)
- Processes 4M+ English-Twi sentence pairs
- Prepares audio data (16kHz mono)
- Creates training manifests

**Run this in Google Colab with GPU runtime for faster processing**

In [ ]:
# Install required packages
!pip install -q librosa soundfile webrtcvad pydub
!pip install -q pandas numpy tqdm
!pip install -q num2words unidecode
!pip install -q audiomentations

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json
from tqdm.auto import tqdm
import zipfile
import tarfile
import shutil

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("Environment: Google Colab")
print(f"Python: {sys.version}")

In [ ]:
# Setup paths
BASE_DIR = Path('/content/drive/MyDrive/voice-lab-data')
RAW_DIR = BASE_DIR / 'raw'
PROCESSED_DIR = BASE_DIR / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base directory: {BASE_DIR}")
print(f"Raw data: {RAW_DIR}")
print(f"Processed data: {PROCESSED_DIR}")

# List available datasets
print("\nAvailable datasets:")
for dataset_dir in RAW_DIR.iterdir():
    if dataset_dir.is_dir():
        # Count files
        file_count = sum(1 for _ in dataset_dir.rglob('*') if _.is_file())
        print(f"  - {dataset_dir.name}: {file_count} files")

## 1. Extract Archives

In [ ]:
# Extract JW300 archive if exists
jw300_dir = RAW_DIR / 'jw300'
jw300_archive = jw300_dir / 'opus-2020-01-08.zip'

if jw300_archive.exists():
    print(f"Extracting JW300 archive ({jw300_archive.stat().st_size / 1024**2:.1f} MB)...")
    
    extract_dir = jw300_dir / 'extracted'
    extract_dir.mkdir(exist_ok=True)
    
    try:
        with zipfile.ZipFile(jw300_archive, 'r') as zip_ref:
            # Extract only Twi-related files
            twi_files = [f for f in zip_ref.namelist() if 'tw' in f.lower()]
            print(f"Found {len(twi_files)} Twi-related files")
            
            for file in tqdm(twi_files[:100], desc="Extracting"):  # Limit for testing
                zip_ref.extract(file, extract_dir)
        
        print("✓ JW300 extraction complete")
    except Exception as e:
        print(f"Error extracting JW300: {e}")
else:
    print("JW300 archive not found")

# Extract Common Voice archive if exists
cv_dir = RAW_DIR / 'common_voice'
cv_archives = list(cv_dir.glob('*.tar.gz'))

if cv_archives:
    cv_archive = cv_archives[0]
    print(f"\nExtracting Common Voice archive ({cv_archive.stat().st_size / 1024**2:.1f} MB)...")
    
    extract_dir = cv_dir / 'extracted'
    extract_dir.mkdir(exist_ok=True)
    
    try:
        with tarfile.open(cv_archive, 'r:gz') as tar:
            # Extract metadata files first
            for member in tar.getmembers():
                if any(x in member.name for x in ['train.tsv', 'test.tsv', 'validated.tsv', 'clips']):
                    if member.isfile() and member.size < 100 * 1024 * 1024:  # Only files < 100MB
                        tar.extract(member, extract_dir)
                        print(f"  Extracted: {member.name}")
        
        print("✓ Common Voice extraction complete")
    except Exception as e:
        print(f"Error extracting Common Voice: {e}")
else:
    print("Common Voice archive not found")

## 2. Process English-Twi 4M+ Sentence Pairs

In [ ]:
# Load and process the 4M+ sentence pairs
michset_dir = RAW_DIR / 'michsethowusu'
pairs_file = michset_dir / 'english_twi_4M_pairs.parquet'

if pairs_file.exists():
    print("Loading 4M+ English-Twi sentence pairs...")
    df_pairs = pd.read_parquet(pairs_file)
    print(f"Loaded {len(df_pairs):,} sentence pairs")
    
    # Filter by similarity score (keep high quality pairs)
    high_quality = df_pairs[df_pairs['similarity'] > 1.0]
    print(f"High quality pairs (similarity > 1.0): {len(high_quality):,}")
    
    # Create train/val/test splits
    from sklearn.model_selection import train_test_split
    
    # First split: 90% train+val, 10% test
    train_val, test = train_test_split(high_quality, test_size=0.1, random_state=42)
    
    # Second split: 90% train, 10% val (of train_val)
    train, val = train_test_split(train_val, test_size=0.1, random_state=42)
    
    print(f"\nSplit sizes:")
    print(f"  Train: {len(train):,} pairs")
    print(f"  Val: {len(val):,} pairs")
    print(f"  Test: {len(test):,} pairs")
    
    # Save splits
    text_dir = PROCESSED_DIR / 'text_data'
    text_dir.mkdir(exist_ok=True)
    
    train.to_parquet(text_dir / 'train_pairs.parquet')
    val.to_parquet(text_dir / 'val_pairs.parquet')
    test.to_parquet(text_dir / 'test_pairs.parquet')
    
    # Also save as TSV for easy viewing
    train.head(1000).to_csv(text_dir / 'train_sample.tsv', sep='\t', index=False)
    
    print("✓ Text data splits saved")
    
    # Create vocabulary
    print("\nCreating Twi vocabulary...")
    twi_texts = train['Twi'].tolist()
    
    # Tokenize and count
    from collections import Counter
    vocab = Counter()
    
    for text in tqdm(twi_texts[:100000], desc="Building vocab"):
        if pd.notna(text):
            words = text.lower().split()
            vocab.update(words)
    
    # Save vocabulary
    with open(text_dir / 'twi_vocab.txt', 'w', encoding='utf-8') as f:
        for word, count in vocab.most_common(50000):
            f.write(f"{word}\t{count}\n")
    
    print(f"✓ Vocabulary created: {len(vocab)} unique words")
    
else:
    print("4M+ pairs file not found. Please check michsethowusu directory.")

## 3. Process Financial Inclusion Audio Data

In [ ]:
# Financial Inclusion dataset configuration
DATASETS = {
    'fante_10': {
        'audio_dir': 'financial_inclusion/fante_10/fisd-fanti-10p/audios',
        'metadata_path': 'financial_inclusion/fante_10/fisd-fanti-10p/data.csv',
        'language': 'fante',
        'split': 'test'
    },
    'fante_90': {
        'audio_dir': 'financial_inclusion/fante_90/fisd-fanti-90p/audios',
        'metadata_path': 'financial_inclusion/fante_90/fisd-fanti-90p/data.csv',
        'language': 'fante',
        'split': 'train'
    },
    'asante_twi_10': {
        'audio_dir': 'financial_inclusion/asante_twi_10/fisd-asanti-twi-10p/audios',
        'metadata_path': 'financial_inclusion/asante_twi_10/fisd-asanti-twi-10p/data.csv',
        'language': 'asante_twi',
        'split': 'test'
    },
    'asante_twi_90': {
        'audio_dir': 'financial_inclusion/asante_twi_90/fisd-asanti-twi-90p/audios',
        'metadata_path': 'financial_inclusion/asante_twi_90/fisd-asanti-twi-90p/data.csv',
        'language': 'asante_twi',
        'split': 'train'
    },
    'akuapem_twi_10': {
        'audio_dir': 'financial_inclusion/akuapem_twi_10/fisd-akuapim-twi-10p/audios',
        'metadata_path': 'financial_inclusion/akuapem_twi_10/fisd-akuapim-twi-10p/data.csv',
        'language': 'akuapem_twi',
        'split': 'test'
    },
    'akuapem_twi_90': {
        'audio_dir': 'financial_inclusion/akuapem_twi_90/fisd-akuapim-twi-90p/audios',
        'metadata_path': 'financial_inclusion/akuapem_twi_90/fisd-akuapim-twi-90p/data.csv',
        'language': 'akuapem_twi',
        'split': 'train'
    }
}

In [ ]:
# Process Financial Inclusion datasets
import librosa
import soundfile as sf

print("Processing Financial Inclusion Audio Datasets")
print("="*50)

all_samples = []
audio_processed_dir = PROCESSED_DIR / 'audio' / 'financial_inclusion'
audio_processed_dir.mkdir(parents=True, exist_ok=True)

for dataset_name, config in DATASETS.items():
    audio_dir = RAW_DIR / config['audio_dir']
    metadata_path = RAW_DIR / config['metadata_path']
    
    if not audio_dir.exists():
        print(f"\n⚠️ {dataset_name}: Audio directory not found")
        continue
    
    # Count audio files
    audio_files = list(audio_dir.glob('*.ogg')) + list(audio_dir.glob('*.wav'))
    print(f"\n📊 {dataset_name}: Found {len(audio_files)} audio files")
    
    # Load metadata if exists
    metadata_df = None
    if metadata_path.exists():
        try:
            metadata_df = pd.read_csv(metadata_path, sep='\t' if metadata_path.suffix == '.tsv' else ',')
            print(f"   ✓ Loaded metadata: {len(metadata_df)} entries")
            
            # Fix audio paths - use just the filename
            if 'Audio Filepath' in metadata_df.columns:
                metadata_df['Audio Filepath'] = metadata_df['Audio Filepath'].apply(lambda x: os.path.basename(x) if pd.notna(x) else x)
            
        except Exception as e:
            print(f"   ⚠️ Error loading metadata: {e}")
    else:
        print(f"   ⚠️ No metadata file at {metadata_path}")
    
    # Process a sample of audio files
    processed_count = 0
    max_samples = 100  # Limit for testing
    
    for audio_file in tqdm(audio_files[:max_samples], desc=f"Processing {dataset_name}"):
        try:
            # Load audio
            audio, sr = librosa.load(audio_file, sr=16000, mono=True)
            
            # Skip if too short or too long
            duration = len(audio) / sr
            if duration < 0.5 or duration > 30:
                continue
            
            # Save processed audio
            output_path = audio_processed_dir / f"{dataset_name}_{audio_file.stem}.wav"
            sf.write(output_path, audio, 16000)
            
            # Get transcription if available
            transcription = ""
            if metadata_df is not None:
                # Try to match by filename
                filename = audio_file.name
                if 'Audio Filepath' in metadata_df.columns:
                    match = metadata_df[metadata_df['Audio Filepath'] == filename]
                    if not match.empty:
                        transcription = str(match.iloc[0].get('Transcription', ''))
            
            # Add to samples
            all_samples.append({
                'audio_path': str(output_path),
                'transcription': transcription,
                'language': config['language'],
                'split': config['split'],
                'duration': duration,
                'dataset': dataset_name
            })
            
            processed_count += 1
            
        except Exception as e:
            continue
    
    print(f"   ✓ Processed {processed_count} audio files")

# Save manifest
if all_samples:
    manifest_df = pd.DataFrame(all_samples)
    manifest_path = PROCESSED_DIR / 'financial_inclusion_manifest.csv'
    manifest_df.to_csv(manifest_path, index=False)
    
    print(f"\n✓ Saved manifest with {len(all_samples)} samples to {manifest_path}")
    print(f"\nDataset statistics:")
    print(manifest_df.groupby(['language', 'split']).size())

## 4. Create Combined Training Manifest

In [ ]:
# Combine all processed data into final training manifests
print("Creating Combined Training Manifests")
print("="*50)

# Collect all manifests
all_manifests = []

# 1. Financial Inclusion manifest
fi_manifest = PROCESSED_DIR / 'financial_inclusion_manifest.csv'
if fi_manifest.exists():
    df_fi = pd.read_csv(fi_manifest)
    all_manifests.append(('Financial Inclusion', df_fi))
    print(f"✓ Financial Inclusion: {len(df_fi)} samples")

# 2. Add other datasets as they are processed
# (BibleTTS, Common Voice, etc.)

# Combine all manifests
if all_manifests:
    combined_df = pd.concat([df for name, df in all_manifests], ignore_index=True)
    
    # Split by train/val/test
    train_df = combined_df[combined_df['split'] == 'train']
    val_df = combined_df[combined_df['split'] == 'val']
    test_df = combined_df[combined_df['split'] == 'test']
    
    # If no val split, create from train
    if len(val_df) == 0 and len(train_df) > 0:
        from sklearn.model_selection import train_test_split
        train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)
    
    print(f"\nFinal splits:")
    print(f"  Train: {len(train_df)} samples")
    print(f"  Val: {len(val_df)} samples")
    print(f"  Test: {len(test_df)} samples")
    
    # Save final manifests
    manifests_dir = PROCESSED_DIR / 'manifests'
    manifests_dir.mkdir(exist_ok=True)
    
    train_df.to_csv(manifests_dir / 'train_manifest.csv', index=False)
    val_df.to_csv(manifests_dir / 'val_manifest.csv', index=False)
    test_df.to_csv(manifests_dir / 'test_manifest.csv', index=False)
    
    # Create JSON manifests for easier loading
    for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        json_manifest = []
        for _, row in split_df.iterrows():
            json_manifest.append({
                'audio_filepath': row['audio_path'],
                'text': row.get('transcription', ''),
                'duration': row.get('duration', 0),
                'language': row.get('language', 'twi')
            })
        
        with open(manifests_dir / f'{split_name}_manifest.json', 'w') as f:
            json.dump(json_manifest, f, indent=2)
    
    print(f"\n✓ Manifests saved to {manifests_dir}")
    
    # Calculate total duration
    total_duration = combined_df['duration'].sum() / 3600  # Convert to hours
    print(f"\n📊 Total audio duration: {total_duration:.2f} hours")
    
    # Language distribution
    print("\n📊 Language distribution:")
    print(combined_df['language'].value_counts())

## 5. Summary and Next Steps

In [ ]:
print("\n" + "="*60)
print("DATA PROCESSING COMPLETE!")
print("="*60)

# Summary statistics
print("\n📊 Processing Summary:")

# Text data
text_dir = PROCESSED_DIR / 'text_data'
if text_dir.exists():
    text_files = list(text_dir.glob('*.parquet'))
    print(f"\n📝 Text Data:")
    for file in text_files:
        df = pd.read_parquet(file)
        print(f"  - {file.stem}: {len(df):,} pairs")

# Audio data
manifests_dir = PROCESSED_DIR / 'manifests'
if manifests_dir.exists():
    manifest_files = list(manifests_dir.glob('*.csv'))
    print(f"\n🎵 Audio Data:")
    for file in manifest_files:
        df = pd.read_csv(file)
        duration = df['duration'].sum() / 3600 if 'duration' in df.columns else 0
        print(f"  - {file.stem}: {len(df)} samples ({duration:.2f} hours)")

print("\n✅ Data is ready for model training!")
print("\n🚀 Next Steps:")
print("1. Train ASR model using audio manifests")
print("2. Train translation model using text pairs")
print("3. Train TTS model using high-quality audio")
print("\n📁 All processed data saved to:")
print(f"   {PROCESSED_DIR}")